In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ sigmoid(z) = p = \frac{e^z}{e^z+1} = \frac{1}{1+e^{-z}} $$
$$ \frac{\partial p}{\partial z} = \frac{e^z}{(e^z+1)^2} = p(1-p) $$


In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from common import T # type: ignore
from approx import approx # type: ignore


def _sigmoid_neg(x: tr.Tensor) -> tr.Tensor:
    """Numerically stable sigmoid for negative inputs."""

    exp = tr.exp(x)
    return exp / (exp + 1)


def _sigmoid_pos(x: tr.Tensor) -> tr.Tensor:
    """Numerically stable sigmoid for positive inputs."""

    return 1 / (1 + tr.exp(-x))


def sigmoid(x: tr.Tensor) -> tr.Tensor:
    """Numerically stable sigmoid function."""

    x = T(x)

    z = tr.empty_like(x)
    pos = x >= 0
    
    z[pos] = _sigmoid_pos(x[pos])
    z[~pos] = _sigmoid_neg(x[~pos])
    
    return z


class SigmoidFunction(autograd.Function):
    """Custom autograd function for the sigmoid function."""

    @staticmethod
    def forward(ctx, z: tr.Tensor) -> tr.Tensor:
        p = sigmoid(z)  
        ctx.save_for_backward(p)
        return p

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, ]:
        (p, ) = ctx.saved_tensors

        df_dz = p * (1 - p)
        assert df_dz.shape == p.shape

        dF_dz = dF_df * df_dz
        assert dF_dz.shape == p.shape

        return (dF_dz, )
    

class Sigmoid(nn.Module):
    """Custom module for the sigmoid function."""
    
    def __init__(self) -> None:
        super().__init__()

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        return SigmoidFunction.apply(x)


def test_sigmoid_1() -> None:
    assert sigmoid(1000) == approx(T(1.0))
    assert sigmoid(-1000) == approx(T(0.0))
    assert sigmoid(0) == approx(T(0.5))
    assert sigmoid(1) == approx(T(0.731), atol=0.001, rtol=0)
    assert sigmoid(-1) == approx(T(0.269), atol=0.001, rtol=0)


def test_sigmoid_2() -> None:
    tr.manual_seed(0)

    samples = 10
    x = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = SigmoidFunction.apply(x1)
    F1 = y1.sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = tr.sigmoid(x2)
    F2 = y2.sum()
    F2.backward()

    assert y1 == approx(y2)
    assert x1.grad == approx(x2.grad)


if __name__ == "__main__":
    test_sigmoid_1()
    test_sigmoid_2()
